# PV1 – Chạy CatVTON trên Colab từ đầu đến cuối (bản dựng lại)

**Cách dùng:** *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*. Không phải sửa ô nào, trừ các công tắc ở ô 0 nếu muốn bỏ bớt việc.

Notebook tự làm lần lượt: kiểm tra GPU → tìm đúng thư mục Drive chung của nhóm → lấy code mới nhất → tạo môi trường Python 3.9 → gom dữ liệu VITON-HD về đĩa Colab → thử 6 cặp → chạy 200 cặp (việc 1.4) → cache demo 60 cặp (1.5) → ảnh ngoài dataset nếu PV5 đã giao (1.7) → mở app Streamlit bằng link công khai (1.6) → bảng kiểm tra cuối.

**Bị ngắt phiên giữa chừng:** mở lại notebook và *Run all* lần nữa. Ảnh đã sinh nằm trên Drive nên được bỏ qua, chỉ chạy tiếp phần còn thiếu.

**Khi một ô báo lỗi:** ô đó in sẵn nguyên nhân và cách sửa. Nếu vẫn không rõ, gửi tên ô (ví dụ "Ô 5") và khoảng 15 dòng cuối.

| Lỗi đã gặp trước đây | Notebook này xử lý thế nào |
|---|---|
| Python 3.13 của Colab không cài được `torch==2.4.0`, `detectron2/_C` chỉ có bản cp39 | Môi trường riêng Python 3.9 bằng `uv`, cài theo file khóa phiên bản `docs/requirements-model.lock.txt` |
| `diffusers` nhánh main, `resolve_revision`, xung đột `huggingface_hub`, `BloomPreTrainedModel` | Ghim cả bộ phiên bản; ô 3 kiểm tra từng phiên bản sau khi cài |
| `MPLBACKEND` của Colab làm lỗi import CatVTON | Đặt `MPLBACKEND=Agg` cho mọi lệnh con |
| `NotImplementedError` do clone phải bản `tryon.py` khung / `git pull` lỗi im lặng | Ô 2 ép code về đúng `origin/main` và kiểm tra `tryon.py` là bản thật |
| `FileNotFoundError ... data/raw/test/image` | Ô 4 tự tìm ảnh ở mọi vị trí PV2 có thể đã để (`data/raw/test/…`, `data/raw/…`, `viton_subset.zip`) và chép về đĩa Colab |
| Colab mở nhầm thư mục `tryon_project` riêng của mình (không phải thư mục chung) | Ô 1 nhận diện thư mục chung theo ID Drive, báo rõ nếu chưa thêm lối tắt |
| `eval_pairs.csv`, `catalog.csv` nằm sai chỗ | Ô 2 tự tìm và chép vào đúng `data/processed/` trong bản clone |
| Nhật ký và ảnh mất khi Colab ngắt | Ảnh, mask, nhật ký đều ghi thẳng lên Drive |
| Streamlit mới không chạy trên Python 3.9; upload ảnh qua link bị lỗi 403 | Dùng `streamlit==1.50.0` trong môi trường 3.9, tắt XSRF khi chạy qua tunnel |

In [ ]:
# Ô 0 – Cấu hình và hàm dùng chung. Chỉ đổi các công tắc RUN_* nếu muốn bỏ bớt việc.
RUN_TRY6 = False        # thử 6 cặp (3 paired + 3 unpaired) trước khi chạy hết
RUN_EVAL200 = False     # việc 1.4: 200 cặp của eval_pairs.csv
RUN_CACHE = False       # việc 1.5: cache demo 6 người x 10 áo
RUN_EXTERNAL = False    # việc 1.7: tự bỏ qua nếu PV5 chưa giao ảnh ngoài dataset
RUN_APP = True         # việc 1.6: mở Streamlit + link công khai

TEAM_FOLDER_ID = "1UvIWdpvKtOjhkWvJPSKMN177HYQIPibB"   # thư mục Drive chung "tryon_project" (của chauthao1603)
TEAM_FOLDER_NAME = "tryon_project"
REPO_URL = "https://github.com/chauthao1603-dotcom/AI-Business-Project.git"
REPO_BRANCH = "main"
CATVTON_URL = "https://github.com/Zheng-Chong/CatVTON.git"
CATVTON_COMMIT = "7818397f25613beedb3d861a34769f607cfcf3b1"   # bản đã chạy được; lấy bản mới nhất nếu không tải được bản này

CONTENT = "/content"
REPO = f"{CONTENT}/AI-Business-Project"
VENV = f"{CONTENT}/venv39"
VENV_PY = f"{VENV}/bin/python"
LOCAL_DATA = f"{CONTENT}/viton_local/test"          # bản sao dữ liệu VITON-HD trên đĩa Colab (nhanh, không phụ thuộc Drive)
EXPECTED = {"torch": "2.4.0", "diffusers": "0.31.0", "transformers": "4.46.3",
            "huggingface_hub": "0.25.2", "numpy": "1.26.4", "streamlit": "1.50.0"}

import csv, glob, os, random, shutil, subprocess, sys, time
from pathlib import Path


class DungLai(Exception):
    '''Lỗi có hướng dẫn sửa; "Run all" dừng tại ô gặp lỗi.'''


def sh(cmd, cwd=None, check=True, quiet=False):
    '''Chạy lệnh, in kết quả theo thời gian thực, trả về mã thoát.'''
    shell = isinstance(cmd, str)
    p = subprocess.Popen(cmd, cwd=cwd, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, executable="/bin/bash" if shell else None)
    tail = []
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-30:]
        if not quiet:
            print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        if quiet:
            print("\n".join(tail))
        raise DungLai(f"Lệnh lỗi (mã {p.returncode}): {cmd if shell else ' '.join(map(str, cmd))}")
    return p.returncode


def count_files(d, suffixes=(".png", ".jpg", ".jpeg")):
    d = Path(d)
    return sum(1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in suffixes) if d.is_dir() else 0


def read_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def write_rows(path, rows, fields):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def show_grid(rows_of_paths, w=192, h=256, title=None):
    '''Hiện lưới ảnh: mỗi phần tử là một hàng gồm các đường dẫn ảnh.'''
    from PIL import Image
    try:
        from IPython.display import display
    except ImportError:
        display = lambda x: None
    rows_of_paths = [r for r in rows_of_paths if r]
    if not rows_of_paths:
        return
    ncol = max(len(r) for r in rows_of_paths)
    sheet = Image.new("RGB", (w * ncol, h * len(rows_of_paths)), "white")
    for i, r in enumerate(rows_of_paths):
        for j, p in enumerate(r):
            if p and Path(p).is_file():
                sheet.paste(Image.open(p).convert("RGB").resize((w, h)), (j * w, i * h))
    if title:
        print(title)
    display(sheet)
    return sheet


print("Ô 0 xong.")


In [ ]:
# Ô 1 – GPU, Drive và thư mục chung của nhóm
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
    raise DungLai("Chưa có GPU. Vào Runtime → Change runtime type → chọn T4 GPU → Save, rồi Run all lại.")
GPU_NAME = gpu.stdout.strip()
print("GPU:", GPU_NAME)

from google.colab import drive
drive.mount("/content/drive")


def _drive_id(path):
    try:
        return os.getxattr(path, "user.drive.id").decode()
    except Exception:
        return None


def is_team_folder(path):
    p = Path(path)
    if not p.is_dir():
        return False
    return (TEAM_FOLDER_ID in os.path.realpath(p) or _drive_id(p) == TEAM_FOLDER_ID
            or (p / "AI-Business-Project-structure-v4.zip").exists())


candidates = [f"/content/drive/.shortcut-targets-by-id/{TEAM_FOLDER_ID}/{TEAM_FOLDER_NAME}",
              f"/content/drive/MyDrive/{TEAM_FOLDER_NAME}"]
candidates += glob.glob(f"/content/drive/MyDrive/*/{TEAM_FOLDER_NAME}")
candidates += glob.glob(f"/content/drive/Shareddrives/*/{TEAM_FOLDER_NAME}")
candidates += glob.glob(f"/content/drive/.shortcut-targets-by-id/*/{TEAM_FOLDER_NAME}")
found = [c for c in dict.fromkeys(candidates) if Path(c).is_dir()]
team = [c for c in found if is_team_folder(c)]
print("Thư mục tên", TEAM_FOLDER_NAME, "tìm thấy:", found or "không có")

if not team:
    msg = ("Không thấy thư mục Drive CHUNG của nhóm. Colab chỉ thấy thư mục được chia sẻ khi nó có lối tắt trong Drive của bạn.\n"
           f"Cách sửa: mở https://drive.google.com/drive/folders/{TEAM_FOLDER_ID} bằng ĐÚNG tài khoản Google đang dùng Colab "
           "→ chuột phải 'tryon_project' → Sắp xếp (Organize) → Thêm lối tắt (Add shortcut) → Drive của tôi (My Drive) → Thêm.\n"
           "Sau đó: Runtime → Restart session, rồi Run all lại.")
    if found:
        msg += ("\nLưu ý: thư mục " + ", ".join(found) + " KHÔNG phải thư mục chung (có thể là bản riêng bạn tạo trước đây). "
                "Đổi tên nó (ví dụ tryon_project_cu) để khỏi nhầm.")
    raise DungLai(msg)

TRYON_ROOT = Path(team[0])
others = [c for c in found if os.path.realpath(c) != os.path.realpath(TRYON_ROOT) and not is_team_folder(c)]
if others:
    print("CẢNH BÁO: còn thư mục trùng tên nhưng không phải thư mục chung:", others,
          "→ notebook KHÔNG dùng chúng. Nên đổi tên để tránh nhầm.")

probe = TRYON_ROOT / "results" / ".pv1_write_test"
try:
    probe.parent.mkdir(parents=True, exist_ok=True)
    probe.write_text("ok")
    probe.unlink()
except OSError as e:
    raise DungLai(f"Không ghi được vào {TRYON_ROOT} ({e}). Nhờ chủ thư mục cấp quyền 'Người chỉnh sửa' (Editor) cho tài khoản này.")

OUT_CATVTON = TRYON_ROOT / "outputs" / "catvton"
OUT_MASKS = TRYON_ROOT / "outputs" / "masks"
OUT_EXTERNAL = TRYON_ROOT / "outputs" / "external"
LOGS = TRYON_ROOT / "results"
CACHE = TRYON_ROOT / "data" / "processed" / "cache"
for d in (OUT_CATVTON, OUT_MASKS, LOGS, CACHE):
    d.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "TRYON_ROOT": str(TRYON_ROOT),          # src/config.py đọc biến này: ảnh và outputs nằm trên Drive
    "HF_HOME": f"{CONTENT}/hf_cache",       # checkpoint tải về đĩa Colab, không chiếm Drive
    "MPLBACKEND": "Agg",                    # tránh lỗi backend matplotlib_inline khi import CatVTON
    "PYTHONUNBUFFERED": "1",
    "TRYON_PRECISION": "fp16",              # T4 không hợp bf16
    "USE_MOCK": "0",
})
print("Thư mục chung:", TRYON_ROOT, "(ghi được)")

In [ ]:
# Ô 2 – Code của nhóm (đúng origin/main) và mã nguồn CatVTON
if not Path(REPO, ".git").is_dir():
    shutil.rmtree(REPO, ignore_errors=True)
    sh(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL, REPO])
sh(["git", "-C", REPO, "fetch", "-q", "origin", REPO_BRANCH])
sh(["git", "-C", REPO, "checkout", "-q", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
sh(["git", "-C", REPO, "reset", "-q", "--hard", f"origin/{REPO_BRANCH}"])     # bỏ mọi sửa tay cũ trong bản clone
sh(["git", "-C", REPO, "log", "-1", "--format=Code nhóm: %h | %ci | %s"])

tryon_src = Path(REPO, "src/tryon.py").read_text(encoding="utf-8")
if "def run_tryon" not in tryon_src or "CatVTONPipeline" not in tryon_src:
    raise DungLai("src/tryon.py trên main không phải bản thật (thiếu CatVTONPipeline). Kiểm tra lại PR của PV1.")
for f in ["scripts/run_batch.py", "scripts/make_cache.py", "src/config.py", "requirements-model.txt", "app.py"]:
    if not Path(REPO, f).is_file():
        raise DungLai(f"Thiếu {f} trên nhánh {REPO_BRANCH}.")

# CSV của PV2: config.py đọc ở data/processed/. PV2 từng để ở data/ -> chép vào đúng chỗ trong bản clone.
for name in ["eval_pairs.csv", "catalog.csv"]:
    dst = Path(REPO, "data/processed", name)
    src = Path(REPO, "data", name)
    if not dst.is_file() and src.is_file():
        shutil.copy(src, dst)
        print(f"Đã chép data/{name} -> data/processed/{name} (chỉ trong bản clone; nhắc PV2 git mv cho gọn).")
EVAL_PAIRS = Path(REPO, "data/processed/eval_pairs.csv")
if not EVAL_PAIRS.is_file():
    raise DungLai("Không thấy eval_pairs.csv (data/processed/ hoặc data/). PV2 cần đẩy file này lên main.")
print("eval_pairs.csv:", len(read_rows(EVAL_PAIRS)), "dòng")

CATVTON = Path(REPO, "CatVTON")


def catvton_ok(d):
    d = Path(d)
    return (d / "model" / "pipeline.py").is_file() and bool(list(d.glob("detectron2/_C.cpython-39-*.so")))


def get_catvton():
    '''Lấy mã nguồn CatVTON, thử lần lượt 3 nguồn. GitHub có lúc từ chối clone ẩn danh từ Colab
    ("could not read Username"), nên nguồn đầu tiên là bản sao nhóm đã để sẵn trên Drive.'''
    drive_copy = TRYON_ROOT / "CatVTON"
    if catvton_ok(drive_copy):
        print("Lấy CatVTON từ Drive:", drive_copy)
        shutil.copytree(drive_copy, CATVTON, ignore=shutil.ignore_patterns(".git", "__pycache__"))
        return
    os.environ["GIT_TERMINAL_PROMPT"] = "0"          # không để git chờ hỏi mật khẩu
    if sh(["git", "clone", "-q", "--depth", "1", CATVTON_URL, str(CATVTON)], check=False) == 0 and catvton_ok(CATVTON):
        print("Lấy CatVTON bằng git clone.")
        return
    shutil.rmtree(CATVTON, ignore_errors=True)
    import tarfile, urllib.request
    url = f"https://codeload.github.com/Zheng-Chong/CatVTON/tar.gz/{CATVTON_COMMIT}"
    tgz = Path(CONTENT, "catvton.tar.gz")
    try:
        urllib.request.urlretrieve(url, tgz)
        with tarfile.open(tgz) as t:
            top = t.getmembers()[0].name.split("/")[0]
            t.extractall(CONTENT)
        shutil.move(str(Path(CONTENT, top)), str(CATVTON))
        print("Lấy CatVTON bằng file nén từ GitHub.")
    except Exception as e:
        print("Tải file nén lỗi:", e)


if not catvton_ok(CATVTON):
    shutil.rmtree(CATVTON, ignore_errors=True)
    get_catvton()
if not catvton_ok(CATVTON):
    raise DungLai("Không lấy được mã nguồn CatVTON từ Drive (tryon_project/CatVTON) lẫn GitHub. "
                  "Kiểm tra thư mục tryon_project/CatVTON trên Drive còn đủ file (model/, detectron2/).")
print("CatVTON:", CATVTON, "| có model/pipeline.py và detectron2/_C (cp39)")
print("Ô 2 xong.")

In [ ]:
# Ô 3 – Môi trường Python 3.9 (lần đầu mỗi phiên mất khoảng 5–8 phút; phiên còn sống thì bỏ qua)
CHECK = ("import json, importlib; "
         f"print(json.dumps({{k: importlib.import_module(k).__version__.split('+')[0] for k in {list(EXPECTED)!r}}}))")


def venv_versions():
    if not Path(VENV_PY).exists():
        return None
    r = subprocess.run([VENV_PY, "-c", CHECK], capture_output=True, text=True)
    if r.returncode != 0:
        return None
    import json
    return json.loads(r.stdout.strip().splitlines()[-1])


have = venv_versions()
if have != EXPECTED:
    print("Tạo môi trường mới. Hiện có:", have)
    shutil.rmtree(VENV, ignore_errors=True)
    sh([sys.executable, "-m", "pip", "install", "-q", "uv"])
    sh(["uv", "venv", "-q", "--python", "3.9", VENV])
    lock = Path(REPO, "docs/requirements-model.lock.txt")
    base = ["uv", "pip", "install", "-q", "--python", VENV_PY, "-r", "requirements-model.txt"]
    if lock.is_file() and sh(base + ["-c", str(lock)], cwd=REPO, check=False) == 0:
        print("Đã cài theo file khóa phiên bản.")
    else:
        print("Cài theo file khóa không được (có thể requirements vừa đổi). Thử cài không khóa...")
        sh(base, cwd=REPO)
    have = venv_versions()

print("Phiên bản:", have)
wrong = {k: (have or {}).get(k) for k, v in EXPECTED.items() if (have or {}).get(k) != v}
if wrong:
    raise DungLai(f"Sai phiên bản {wrong}, cần {EXPECTED}. Xóa {VENV} (!rm -rf {VENV}) rồi chạy lại ô này; "
                  "nếu vẫn sai, gửi 15 dòng cuối của ô.")
print("Ô 3 xong.")

In [ ]:
# Ô 3b – Kiểm tra import CatVTON trên GPU (bắt các lỗi detectron2, matplotlib, diffusers ngay từ đầu)
probe = '''
import torch
assert torch.cuda.is_available(), "torch không thấy GPU"
import model.pipeline, model.cloth_masker
from model.DensePose import DensePose
from model.SCHP import SCHP
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0), "| import CatVTON OK")
try:
    from detectron2 import _C
    print("detectron2._C: nạp được")
except ImportError:
    # File _C kèm repo CatVTON được biên dịch cho torch cũ, không khớp torch 2.4.0. Không sao: _C chỉ phục vụ
    # deformable conv (detectron2 tự bỏ qua khi thiếu), cấu hình DensePose R_50_FPN_s1x của CatVTON không dùng tới.
    print("detectron2._C: không nạp được với torch 2.4.0 (bình thường, AutoMasker không cần; ô 6 sẽ kiểm tra thật)")
'''
if sh([VENV_PY, "-c", probe], cwd=str(CATVTON), check=False) != 0:
    raise DungLai("Import CatVTON lỗi. Gửi 15 dòng cuối phía trên. (Đã đặt MPLBACKEND=Agg; nếu lỗi nhắc tới detectron2 "
                  "thì vẫn chạy được 200 cặp bằng mask có sẵn, chỉ cache demo và app cần AutoMasker.)")
print("Ô 3b xong.")

In [ ]:
# Ô 4 – Gom dữ liệu VITON-HD cần cho 200 cặp về đĩa Colab
rows = read_rows(EVAL_PAIRS)
need = {"image": {r["person"] for r in rows}, "cloth": {r["cloth"] for r in rows},
        "agnostic-mask": {r["person"] for r in rows}}
NAMES = {"image": ["{s}.jpg", "{s}.jpeg", "{s}.png"], "cloth": ["{s}.jpg", "{s}.jpeg", "{s}.png"],
         "agnostic-mask": ["{s}_mask.png"]}


def local_has(sub, s):
    return any(Path(LOCAL_DATA, sub, n.format(s=s)).is_file() for n in NAMES[sub])


def missing_now():
    return {sub: sorted(s for s in stems if not local_has(sub, s)) for sub, stems in need.items()}


raw = TRYON_ROOT / "data" / "raw"
roots = [raw / "test", raw, raw / "viton_subset" / "test", raw / "viton_subset", TRYON_ROOT / "viton_subset" / "test"]
for sub in need:
    Path(LOCAL_DATA, sub).mkdir(parents=True, exist_ok=True)
    for root in roots:
        d = root / sub
        if not d.is_dir():
            continue
        got = 0
        for s in sorted(need[sub]):
            if local_has(sub, s):
                continue
            for n in NAMES[sub]:
                f = d / n.format(s=s)
                if f.is_file():
                    shutil.copy(f, Path(LOCAL_DATA, sub, f.name))
                    got += 1
                    break
        if got:
            print(f"{sub}: lấy {got} file từ {d}")

miss = missing_now()
if any(miss.values()):
    zips = [z for z in [TRYON_ROOT / "viton_subset.zip", raw / "viton_subset.zip"] if z.is_file()]
    for z in zips:
        import zipfile
        print(f"Còn thiếu {sum(map(len, miss.values()))} file, lấy từ {z} ...")
        local_zip = Path(CONTENT, "viton_subset.zip")
        if not local_zip.is_file() or local_zip.stat().st_size != z.stat().st_size:
            shutil.copy(z, local_zip)
        want = {sub: {n.format(s=s) for s in miss[sub] for n in NAMES[sub]} for sub in miss}
        with zipfile.ZipFile(local_zip) as zf:
            for info in zf.infolist():
                parts = Path(info.filename).parts
                if len(parts) >= 2 and parts[-2] in want and parts[-1] in want[parts[-2]]:
                    with zf.open(info) as src, open(Path(LOCAL_DATA, parts[-2], parts[-1]), "wb") as dst:
                        shutil.copyfileobj(src, dst)
        miss = missing_now()
        if not any(miss.values()):
            break

for sub in need:
    print(f"{sub:14s}: có {len(need[sub]) - len(miss[sub])}/{len(need[sub])}")

bad_ids = {r["id"] for r in rows if r["person"] in miss["image"] or r["cloth"] in miss["cloth"]
           or r["person"] in miss["agnostic-mask"]}
READY = [r for r in rows if r["id"] not in bad_ids]
PAIRS_READY = Path(CONTENT, "pairs_ready.csv")
write_rows(PAIRS_READY, READY, ["id", "person", "cloth", "mode"])
if bad_ids:
    write_rows(LOGS / "pv1_missing_inputs.csv", [r for r in rows if r["id"] in bad_ids], ["id", "person", "cloth", "mode"])
    print(f"CẢNH BÁO: {len(bad_ids)} cặp thiếu ảnh/mask, tạm bỏ qua. Danh sách: {LOGS / 'pv1_missing_inputs.csv'} (gửi PV2).")
if not READY:
    raise DungLai("Không cặp nào đủ dữ liệu. PV2 cần đưa ảnh VITON-HD lên Drive: tryon_project/data/raw/test/{image,cloth,agnostic-mask} "
                  "hoặc file viton_subset.zip ở tryon_project/.")

# Kiểm tra nhanh chuẩn dữ liệu trên 3 mẫu: ảnh 768x1024, mask nhị phân, vùng trắng là vùng áo (khoảng 10–60% ảnh)
import numpy as np
from PIL import Image
for r in READY[:3]:
    im = Image.open(next(Path(LOCAL_DATA, "image").glob(r["person"] + ".*")))
    m = np.asarray(Image.open(Path(LOCAL_DATA, "agnostic-mask", r["person"] + "_mask.png")).convert("L"))
    white = (m > 127).mean() * 100
    print(f"  {r['person']}: ảnh {im.size}, mask trắng {white:.0f}%")
    if not 5 < white < 70:
        print("  CẢNH BÁO: tỉ lệ vùng trắng bất thường, kiểm tra lại chiều mask (config.AGNOSTIC_REGION_IS_WHITE).")
print(f"Ô 4 xong: {len(READY)}/{len(rows)} cặp sẵn sàng "
      f"({sum(r['mode'] == 'paired' for r in READY)} paired, {sum(r['mode'] == 'unpaired' for r in READY)} unpaired).")

In [ ]:
# Ô 5 – Hàm chạy batch dùng chung cho các ô sau
def run_batch(pairs, out, tag, mode=None, mask_source="dataset", person_dir=None, cloth_dir=None,
              mask_dir=None, masks=None, max_err=5):
    cmd = [VENV_PY, "-m", "scripts.run_batch", "--pairs", str(pairs),
           "--person_dir", str(person_dir or Path(LOCAL_DATA, "image")),
           "--cloth_dir", str(cloth_dir or Path(LOCAL_DATA, "cloth")),
           "--mask_dir", str(mask_dir or Path(LOCAL_DATA, "agnostic-mask")),
           "--mask_source", mask_source, "--out", str(out), "--masks", str(masks or OUT_MASKS),
           "--log_dir", str(LOGS), "--tag", tag, "--max_consecutive_errors", str(max_err)]
    if mode:
        cmd += ["--mode", mode]
    code_ = sh(cmd, cwd=REPO, check=False)
    err = LOGS / f"errors_{tag}.csv"
    if code_ != 0:
        if err.is_file():
            print("--- 5 lỗi gần nhất trong", err, "---")
            print("\n".join(err.read_text(encoding="utf-8").splitlines()[-5:]))
        raise DungLai(f"run_batch dừng vì nhiều cặp liên tiếp lỗi (tag {tag}). Xem các dòng lỗi ngay phía trên.")


def done_ids(out):
    return {p.stem for p in Path(out).glob("*.png")}


def timing_stats(tags):
    '''Giây/ảnh và VRAM từ các file timing_log. Bỏ các lần gồm thời gian nạp mô hình (> 2.5 lần trung vị).'''
    secs, vram = [], []
    for t in tags:
        f = LOGS / f"timing_log_{t}.csv"
        if f.is_file():
            for r in read_rows(f):
                secs.append(float(r["seconds"]))
                if r.get("peak_vram_gb"):
                    vram.append(float(r["peak_vram_gb"]))
    if not secs:
        return None
    med = sorted(secs)[len(secs) // 2]
    steady = [s for s in secs if s <= 2.5 * med] or secs
    return {"n": len(secs), "sec_per_img": sum(steady) / len(steady), "first_run_sec": max(secs),
            "peak_vram_gb": max(vram) if vram else None}

print("Ô 5 xong.")

In [ ]:
# Ô 6 – Thử 6 cặp (3 paired + 3 unpaired). Lần đầu mất thêm vài phút để tải checkpoint (vài GB).
if RUN_TRY6:
    try6 = [r for r in READY if r["mode"] == "paired"][:3] + [r for r in READY if r["mode"] == "unpaired"][:3]
    write_rows(Path(CONTENT, "pairs_try6.csv"), try6, ["id", "person", "cloth", "mode"])
    run_batch(Path(CONTENT, "pairs_try6.csv"), OUT_CATVTON, tag="try6", max_err=3)

    lacking = [r["id"] for r in try6 if r["id"] not in done_ids(OUT_CATVTON)]
    if lacking:
        raise DungLai(f"Thử 6 cặp chưa đạt, thiếu {lacking}. Xem {LOGS / 'errors_try6.csv'}.")
    from PIL import Image
    import numpy as np
    for r in try6:
        res = Image.open(OUT_CATVTON / f"{r['id']}.png")
        m = np.asarray(Image.open(OUT_MASKS / f"{r['person']}.png"))
        assert res.size == (768, 1024) and res.mode == "RGB", (r["id"], res.size, res.mode)
        assert set(np.unique(m)) <= {0, 255}, f"mask {r['person']} không nhị phân"
    show_grid([[next(Path(LOCAL_DATA, "image").glob(r["person"] + ".*")),
                next(Path(LOCAL_DATA, "cloth").glob(r["cloth"] + ".*")),
                OUT_CATVTON / f"{r['id']}.png"] for r in try6],
              title="Mỗi hàng: ảnh người | ảnh áo | kết quả. 3 hàng đầu là paired (phải gần giống ảnh gốc).")
    print("Số liệu:", timing_stats(["try6"]))
    print("Ô 6 đạt: 6 ảnh 768x1024 RGB, mask nhị phân. Nhìn lưới ảnh: đúng người, đúng màu và họa tiết áo, không méo mặt/tay.")

    # Kiểm tra AutoMasker (DensePose + SCHP) bằng 1 cặp: cache demo (ô 9) và app (ô 11) cần nó, 200 cặp thì không.
    r0 = try6[-1]
    AUTOMASK_OK = sh([VENV_PY, "-m", "scripts.smoke_tryon",
                      "--person", str(next(Path(LOCAL_DATA, "image").glob(r0["person"] + ".*"))),
                      "--garment", str(next(Path(LOCAL_DATA, "cloth").glob(r0["cloth"] + ".*"))),
                      "--out", f"{CONTENT}/smoke_auto"], cwd=REPO, check=False) == 0
    if AUTOMASK_OK:
        show_grid([[p for p in sorted(Path(CONTENT, "smoke_auto").glob("*.png"))]],
                  title="AutoMasker: ảnh kết quả và mask tự tạo (mask phải phủ đúng vùng áo thân trên).")
        print("AutoMasker chạy được.")
    else:
        print("CẢNH BÁO: AutoMasker lỗi (xem dòng lỗi phía trên, gửi cho PV1). Ô 7 vẫn chạy được vì dùng mask có sẵn; "
              "ô 9 (cache) sẽ dừng, app ở ô 11 chỉ dùng được các cặp đã có trong cache.")
else:
    print("Bỏ qua (RUN_TRY6 = False).")

In [ ]:
# Ô 7 – Việc 1.4: chạy đủ các cặp của eval_pairs.csv (paired trước vì PV3 cần trước cho MSE/SSIM/LPIPS)
# Khoảng 200 x (giây/ảnh ở ô 6). Bị ngắt thì Run all lại: cặp đã có ảnh trên Drive được bỏ qua.
if RUN_EVAL200:
    for mode in ["paired", "unpaired"]:
        print(f"===== {mode} =====")
        run_batch(PAIRS_READY, OUT_CATVTON, tag="catvton", mode=mode)
    left = [r for r in READY if r["id"] not in done_ids(OUT_CATVTON)]
    if left:                                   # chạy lại một lượt cho cặp lỗi lẻ tẻ (thường là lỗi tạm thời)
        print(f"Chạy lại {len(left)} cặp còn thiếu...")
        run_batch(PAIRS_READY, OUT_CATVTON, tag="catvton")
    left = [r["id"] for r in READY if r["id"] not in done_ids(OUT_CATVTON)]
    print(f"Ô 7: có {len(READY) - len(left)}/{len(READY)} ảnh trong {OUT_CATVTON}."
          + (f" Còn thiếu {len(left)}: {left[:10]}... xem {LOGS / 'errors_catvton.csv'}" if left else " Đủ."))
else:
    print("Bỏ qua (RUN_EVAL200 = False).")

In [ ]:
# Ô 8 – Số liệu cho báo cáo + xem ngẫu nhiên 10 ảnh
st = timing_stats(["try6", "catvton"])
if st:
    summary = {"gpu": GPU_NAME, "so_anh_do": st["n"], "giay_moi_anh": round(st["sec_per_img"], 1),
               "lan_dau_gom_nap_mo_hinh_giay": round(st["first_run_sec"], 1),
               "vram_dinh_gb": st["peak_vram_gb"], "uoc_tinh_200_cap_phut": round(st["sec_per_img"] * 200 / 60),
               "so_buoc": 50, "guidance": 2.5, "seed": 42, "do_chinh_xac": "fp16"}
    write_rows(LOGS / "pv1_timing_summary.csv", [summary], list(summary))
    for k, v in summary.items():
        print(f"{k:32s} {v}")
    print("Đã lưu", LOGS / "pv1_timing_summary.csv")
have_ids = sorted(done_ids(OUT_CATVTON) & {r["id"] for r in READY})
pick = random.Random(42).sample(have_ids, min(10, len(have_ids)))
by_id = {r["id"]: r for r in READY}
show_grid([[next(Path(LOCAL_DATA, "image").glob(by_id[i]["person"] + ".*")),
            next(Path(LOCAL_DATA, "cloth").glob(by_id[i]["cloth"] + ".*")),
            OUT_CATVTON / f"{i}.png"] for i in pick], w=144, h=192,
          title="10 cặp ngẫu nhiên (người | áo | kết quả). Ảnh lỗi rõ ràng thì ghi id lại để báo PV3 (failure cases).")

In [ ]:
# Ô 9 – Việc 1.5: cache demo 6 người x 10 áo (áo = 10 sản phẩm đầu của catalog.csv để app dùng lại đúng cache)
DEMO_P = TRYON_ROOT / "data/processed/demo/persons"
DEMO_G = TRYON_ROOT / "data/processed/demo/garments"
SHOP_G = TRYON_ROOT / "data/processed/garments"


def ensure_demo_assets():
    '''Đưa ảnh demo và ảnh áo cửa hàng lên Drive nếu còn trống (không ghi đè thứ PV2 đã để).'''
    catalog = Path(REPO, "data/processed/catalog.csv")
    files = [r["image_file"] for r in read_rows(catalog)] if catalog.is_file() else []
    repo_g = [Path(REPO, "data/garments"), Path(REPO, "data/processed/garments"), Path(REPO, "data/processed/demo/garments")]

    def find_g(name):
        for d in [SHOP_G] + repo_g:
            if (d / name).is_file():
                return d / name
        return None

    SHOP_G.mkdir(parents=True, exist_ok=True)
    for name in files:                                   # ảnh áo cho trang Cửa hàng của app
        if not (SHOP_G / name).is_file() and find_g(name):
            shutil.copy(find_g(name), SHOP_G / name)
    p_local = sorted(Path(REPO, "data/processed/demo/persons").glob("*.jpg"))
    if count_files(DEMO_P) < len(p_local):
        DEMO_P.mkdir(parents=True, exist_ok=True)
        for f in p_local:
            shutil.copy(f, DEMO_P / f.name)
    if count_files(DEMO_G) < 10:
        DEMO_G.mkdir(parents=True, exist_ok=True)
        for name in files[:10]:
            if find_g(name):
                shutil.copy(find_g(name), DEMO_G / name)
    print(f"Drive: {count_files(DEMO_P)} ảnh người demo, {count_files(DEMO_G)} áo demo, {count_files(SHOP_G)} áo cửa hàng.")


if RUN_CACHE:
    if globals().get("AUTOMASK_OK") is False:
        raise DungLai("AutoMasker lỗi ở ô 6 nên chưa sinh được cache. Gửi PV1 phần lỗi của ô 6.")
    ensure_demo_assets()
    if count_files(DEMO_P) == 0 or count_files(DEMO_G) == 0:
        raise DungLai(f"Chưa có ảnh demo trên Drive ({DEMO_P}, {DEMO_G}) và repo cũng không còn. Nhờ PV2 đưa bộ demo lên Drive.")
    code_ = sh([VENV_PY, "-m", "scripts.make_cache", "--persons", str(DEMO_P), "--garments", str(DEMO_G),
                "--out", str(CACHE), "--log_dir", str(LOGS)], cwd=REPO, check=False)
    if code_ != 0:
        err = LOGS / "errors_cache.csv"
        print("\n".join(err.read_text(encoding="utf-8").splitlines()[-5:]) if err.is_file() else "")
        raise DungLai("make_cache dừng vì nhiều cặp liên tiếp lỗi. Xem dòng lỗi phía trên.")
    n_img = len([p for p in CACHE.glob("*.png") if not p.name.endswith("_mask.png")])
    n_exp = count_files(DEMO_P) * count_files(DEMO_G)
    print(f"Ô 9: cache có {n_img}/{n_exp} cặp (mỗi cặp 2 file: ảnh + _mask).")
    persons = sorted(p.stem for p in DEMO_P.iterdir() if p.is_file())
    garments = sorted(p.stem for p in DEMO_G.iterdir() if p.is_file())
    show_grid([[CACHE / f"{p}__{g}.png" for g in garments] for p in persons], w=96, h=128,
              title="Cache demo: mỗi hàng 1 người, mỗi cột 1 áo. Gửi PV2 lọc cặp lỗi (demo_review.csv).")
else:
    print("Bỏ qua (RUN_CACHE = False).")


In [ ]:
# Ô 10 – Việc 1.7: ảnh ngoài dataset (chỉ chạy khi PV5 đã giao external_pairs.csv và ảnh trên Drive)
EXT_PAIRS = Path(REPO, "data/processed/external_pairs.csv")
EXT_P = TRYON_ROOT / "data/processed/external/persons"
EXT_G = TRYON_ROOT / "data/processed/external/garments"
if not RUN_EXTERNAL:
    print("Bỏ qua (RUN_EXTERNAL = False).")
elif not EXT_PAIRS.is_file() or count_files(EXT_P) == 0 or count_files(EXT_G) == 0:
    print(f"Bỏ qua: PV5 chưa giao (cần {EXT_PAIRS.relative_to(REPO)} trên main, ảnh trong {EXT_P} và {EXT_G}).")
else:
    run_batch(EXT_PAIRS, OUT_EXTERNAL, tag="catvton_external", mask_source="auto", person_dir=EXT_P, cloth_dir=EXT_G)
    print(f"Ô 10: {count_files(OUT_EXTERNAL)} ảnh trong {OUT_EXTERNAL}.")

In [ ]:
# Ô 11 – Việc 1.6: mở app Streamlit (môi trường Python 3.9, mô hình thật) qua link công khai cloudflared
APP_LOG, CF_LOG = Path(CONTENT, "streamlit.log"), Path(CONTENT, "cloudflared.log")
CLOUDFLARED = Path(CONTENT, "cloudflared")


def stop_app():
    subprocess.run("pkill -f 'streamlit run' ; pkill -f 'cloudflared tunnel'", shell=True)


if RUN_APP:
    import re, urllib.request
    ensure_demo_assets()
    stop_app()
    time.sleep(2)
    if not CLOUDFLARED.is_file():
        sh(["wget", "-q", "-O", str(CLOUDFLARED),
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
        CLOUDFLARED.chmod(0o755)
    env = dict(os.environ, USE_MOCK="0")
    subprocess.Popen([f"{VENV}/bin/streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                      "--server.enableXsrfProtection", "false", "--server.enableCORS", "false",
                      "--server.fileWatcherType", "none", "--browser.gatherUsageStats", "false"],
                     cwd=REPO, env=env, stdout=open(APP_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read().strip() == b"ok":
                break
        except Exception:
            pass
        time.sleep(2)
    else:
        print(APP_LOG.read_text()[-3000:])
        raise DungLai("Streamlit không khởi động được. Xem log phía trên.")
    subprocess.Popen([str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
                     stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    url = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", CF_LOG.read_text() if CF_LOG.exists() else "")
        if m:
            url = m.group(0)
            break
        time.sleep(2)
    if not url:
        print(CF_LOG.read_text()[-2000:])
        raise DungLai("Không lấy được link cloudflared. Chạy lại ô này.")
    print("LINK APP:", url)
    print("Link sống khi tab Colab còn mở. Cặp có trong cache hiện ngay; ảnh mới lần đầu mất 1–2 phút để nạp mô hình, "
          "các lần sau khoảng bằng giây/ảnh ở ô 8. Đừng chạy ô 7/9 cùng lúc với app (T4 không đủ bộ nhớ cho hai mô hình).")
    print("Log app:", APP_LOG, "| tắt app: chạy ô 12.")
else:
    print("Bỏ qua (RUN_APP = False).")

In [ ]:
# Ô 12 – Tắt app (chạy khi cần GPU cho việc khác)
stop_app()
print("Đã tắt Streamlit và cloudflared.")

In [ ]:
# Ô 13 – Bảng kiểm tra cuối (chụp màn hình gửi nhóm)
n_cache = len([p for p in CACHE.glob("*.png") if not p.name.endswith("_mask.png")])
n_eval = len(done_ids(OUT_CATVTON) & {r["id"] for r in rows})
checks = [
    ("1.4 Ảnh CatVTON (outputs/catvton)", n_eval, len(rows), "PV3"),
    ("1.4 Mask đã dùng (outputs/masks)", count_files(OUT_MASKS), len({r["person"] for r in rows}), "PV2, PV3"),
    ("1.4 timing_log_catvton.csv", int((LOGS / "timing_log_catvton.csv").is_file()), 1, "PV3 (báo cáo)"),
    ("1.5 Cache demo (data/processed/cache)", n_cache, 60, "PV2, PV4"),
    ("1.7 Ảnh ngoài dataset (outputs/external)", count_files(OUT_EXTERNAL), "khi PV5 giao", "PV3"),
]
for name, have, want, who in checks:
    ok = "✓" if isinstance(want, int) and have >= want else ("○" if not isinstance(want, int) else "✗")
    print(f"{ok} {name:42s} {have} / {want}   → giao {who}")
print("\nNhật ký trên Drive:", sorted(p.name for p in LOGS.glob("*.csv")))
print("Tải results/timing_log_catvton.csv, errors_catvton.csv, pv1_timing_summary.csv về repo (thư mục results/) rồi commit.")